In [1]:
# Cell 1: clone repo
!git clone https://github.com/nnzhan/Graph-WaveNet.git
%cd Graph-WaveNet

Cloning into 'Graph-WaveNet'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 53 (delta 12), reused 12 (delta 12), pack-reused 33 (from 1)
Receiving objects: 100% (53/53), 267.08 KiB | 14.06 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Graph-WaveNet


In [2]:
# Clone DCRNN để lấy file adj
!git clone https://github.com/liyaguang/DCRNN.git

# Kiểm tra file có ở đó không
!ls DCRNN/data/sensor_graph/

Cloning into 'DCRNN'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 334 (delta 51), reused 34 (delta 34), pack-reused 268 (from 2)
Receiving objects: 100% (334/334), 127.90 MiB | 35.61 MiB/s, done.
Resolving deltas: 100% (153/153), done.
adj_mx_bay.pkl		graph_sensor_ids.txt
adj_mx.pkl		graph_sensor_locations_bay.csv
distances_bay_2017.csv	graph_sensor_locations.csv
distances_la_2012.csv


In [3]:
# Copy file adj vào đúng chỗ Graph-WaveNet cần
!mkdir -p /kaggle/working/Graph-WaveNet/data/sensor_graph

# METR-LA
!cp DCRNN/data/sensor_graph/adj_mx.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx.pkl

# PEMS-BAY
!cp DCRNN/data/sensor_graph/adj_mx_bay.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx_bay.pkl

In [4]:
# Kiểm tra lại
!ls /kaggle/working/Graph-WaveNet/data/sensor_graph/

adj_mx_bay.pkl	adj_mx.pkl


In [5]:
%%writefile /kaggle/working/Graph-WaveNet/model.py
"""
model_fixed.py  —  Enhanced Graph WaveNet
Drop-in replacement for the original model.py from nnzhan/Graph-WaveNet

Fixes applied (theo roadmap):
  FIX-01  Cross-layer Skip Attention  (thay ST-Attention song song GCN)
  FIX-02  Causal Window Attention TCN (thay full Transformer / dilated conv thuần)
  FIX-03  Per-module LR param_groups  (helper build_optimizer)

Không thay đổi:
  - Interface  gwnet(device, num_nodes, …)  giữ nguyên
  - forward(input, adj) signature giống bản gốc
  - Adaptive adjacency matrix (E1, E2) giữ nguyên
  - Top-k sparse adj (nếu bạn đã có) không bị ảnh hưởng

Usage (Kaggle notebook):
    from model_fixed import gwnet, build_optimizer
    model = gwnet(device, num_nodes=207, …)          # METR-LA: 207, PEMS-BAY: 325
    optimizer = build_optimizer(model, base_lr=0.001)
    # training loop giống hệt bản gốc
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ─────────────────────────────────────────────
#  FIX-02 helpers: Causal Window Attention TCN
# ─────────────────────────────────────────────

def _causal_window_mask(seq_len: int, window: int, device) -> torch.Tensor:
    """
    Boolean mask (True = IGNORE) cho causal + local window attention.
    Position i chỉ attend được [max(0, i-window+1) … i].
    Shape: (seq_len, seq_len)
    """
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool, device=device)
    for i in range(seq_len):
        lo = max(0, i - window + 1)
        mask[i, lo : i + 1] = False          # False = attend được
    return mask


class RelativePositionalEncoding(nn.Module):
    """
    Học relative PE (không dùng absolute sinusoidal — tránh leakage future).
    Bias được thêm vào attention logits trước softmax.
    """
    def __init__(self, num_heads: int, max_len: int = 64):
        super().__init__()
        self.num_heads = num_heads
        # Bias cho khoảng cách 0..max_len-1
        self.rel_bias = nn.Embedding(max_len, num_heads)
        nn.init.zeros_(self.rel_bias.weight)

    def forward(self, seq_len: int) -> torch.Tensor:
        """Returns (seq_len, seq_len, num_heads) additive bias."""
        device = self.rel_bias.weight.device
        idx = torch.arange(seq_len, device=device)
        # relative distance (clamp âm → 0 để causal)
        dist = (idx.unsqueeze(1) - idx.unsqueeze(0)).clamp(min=0)
        dist = dist.clamp(max=self.rel_bias.num_embeddings - 1)
        bias = self.rel_bias(dist)            # (T, T, H)
        return bias.permute(2, 0, 1)          # (H, T, T)


class CausalWindowAttnTCN(nn.Module):
    """
    FIX-02: Thay thế dilated causal conv + (optional) Transformer bằng
    Causal Window Multi-Head Attention.

    - window_size = 2 * dilation_factor  →  receptive field tương đương dilated conv
    - Causal mask đảm bảo không attend future
    - Relative PE thay absolute PE
    - Output shape = input shape để plug-in giữa GCN layers

    Args:
        in_channels (int): số channels đầu vào (C)
        out_channels (int): số channels đầu ra  (C)
        kernel_size (int): dilation factor (default 2, như bản gốc)
        num_heads (int): số attention heads
        dropout (float): dropout sau attention
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.window_size = max(2, 2 * kernel_size)   # local window

        # Project vào head-dim
        head_dim = max(out_channels // num_heads, 1)
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.scale = math.sqrt(head_dim)

        self.q_proj = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(num_heads * head_dim, out_channels)

        self.rel_pe = RelativePositionalEncoding(num_heads)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_channels)

        # Gate tương tự gated activation bản gốc
        self.gate = nn.Linear(out_channels, out_channels)

        # Residual projection nếu in != out
        self.residual_proj = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, C, N, T)   — graph WaveNet standard layout
        returns: (B, out_channels, N, T-1)  (drop last step để match dilated conv)
        """
        B, C, N, T = x.shape

        # Transpose → (B*N, T, C) để dùng Linear
        x_bn = x.permute(0, 2, 3, 1).reshape(B * N, T, C)  # (BN, T, C)

        Q = self.q_proj(x_bn)   # (BN, T, H*D)
        K = self.k_proj(x_bn)
        V = self.v_proj(x_bn)

        H, D = self.num_heads, self.head_dim
        Q = Q.view(B * N, T, H, D).transpose(1, 2)  # (BN, H, T, D)
        K = K.view(B * N, T, H, D).transpose(1, 2)
        V = V.view(B * N, T, H, D).transpose(1, 2)

        # Attention scores + relative PE bias
        attn = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (BN, H, T, T)
        rel_bias = self.rel_pe(T)                                  # (H, T, T)
        attn = attn + rel_bias.unsqueeze(0)

        # Causal + window mask
        mask = _causal_window_mask(T, self.window_size, x.device)  # (T, T)
        attn = attn.masked_fill(mask, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)   # handle all-masked rows
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)              # (BN, H, T, D)
        out = out.transpose(1, 2).reshape(B * N, T, H * D)
        out = self.out_proj(out)                 # (BN, T, out_C)

        # Gated activation (tương tự tanh * sigmoid bản gốc)
        out = out * torch.sigmoid(self.gate(out))
        out = self.norm(out)

        # Reshape về (B, out_C, N, T)
        out = out.view(B, N, T, -1).permute(0, 3, 1, 2)

        # Residual — drop first timestep để match dilated conv output T→T-1
        res = self.residual_proj(x)
        out = out[:, :, :, 1:] + res[:, :, :, 1:]   # (B, out_C, N, T-1)
        return out


# ─────────────────────────────────────────────
#  FIX-01: Cross-layer Skip Attention
# ─────────────────────────────────────────────

class SkipAggregationAttn(nn.Module):
    """
    FIX-01: Thay Spatial-Temporal Attention song song GCN.
    Aggregate K skip tensors bằng attention qua chiều layer, KHÔNG can thiệp GCN.

    Dùng ở cuối forward() để combine tất cả skip connections.

    Args:
        channels (int): số channels của mỗi skip tensor
        num_heads (int): attention heads
    """
    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        # Đảm bảo channels chia hết cho num_heads
        while channels % num_heads != 0 and num_heads > 1:
            num_heads //= 2
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True,
            dropout=0.1,
        )
        self.norm = nn.LayerNorm(channels)

    def forward(self, skip_list: list) -> torch.Tensor:
        """
        skip_list: list of K tensors, each (B, C, N, T_out)
                   T_out nên là 1 sau khi reduce (sum/mean hoặc final step)
        returns  : (B, C, N, 1)
        """
        # Mỗi skip: reduce T → 1 bằng mean nếu chưa
        processed = []
        for s in skip_list:
            if s.dim() == 4:
                s = s.mean(dim=-1, keepdim=True)   # (B, C, N, 1)
            processed.append(s)

        B, C, N, _ = processed[0].shape
        K = len(processed)

        # Stack → (B*N, K, C)
        stacked = torch.stack(processed, dim=2)   # (B, C, N, K) — wait, careful
        # → reorder to (B, N, K, C) then reshape (B*N, K, C)
        stacked = stacked.squeeze(-1).permute(0, 2, 3, 1)  # (B, N, K, C) ... hmm

        # Correct stacking: each elem is (B, C, N) after squeeze
        elems = [p.squeeze(-1).permute(0, 2, 1) for p in processed]  # each (B, N, C)
        stacked = torch.stack(elems, dim=2)          # (B, N, K, C)
        stacked_2d = stacked.view(B * N, K, C)       # (BN, K, C)

        # Self-attention across K layers
        attended, _ = self.attn(stacked_2d, stacked_2d, stacked_2d)
        attended = self.norm(attended + stacked_2d)

        # Aggregate across K → (BN, C) bằng mean
        out = attended.mean(dim=1)                   # (BN, C)
        out = out.view(B, N, C).permute(0, 2, 1)     # (B, C, N)
        return out.unsqueeze(-1)                      # (B, C, N, 1)


# ─────────────────────────────────────────────
#  GCN layer (giữ nguyên từ bản gốc)
# ─────────────────────────────────────────────

class nconv(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, A):
        x = torch.einsum('ncvl,vw->ncwl', (x, A))
        return x.contiguous()


class linear(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.mlp = nn.Conv2d(c_in, c_out, kernel_size=(1, 1), padding=(0, 0),
                             stride=(1, 1), bias=True)

    def forward(self, x):
        return self.mlp(x)


class gcn(nn.Module):
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super().__init__()
        self.nconv = nconv()
        c_in_actual = (order * support_len + 1) * c_in
        self.mlp = linear(c_in_actual, c_out)
        self.dropout = dropout
        self.order = order

    def forward(self, x, support):
        out = [x]
        for a in support:
            x1 = self.nconv(x, a)
            out.append(x1)
            for _ in range(2, self.order + 1):
                x2 = self.nconv(x1, a)
                out.append(x2)
                x1 = x2
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h


# ─────────────────────────────────────────────
#  Main model: gwnet (Fixed)
# ─────────────────────────────────────────────

class gwnet(nn.Module):
    """
    Enhanced Graph WaveNet với FIX-01, FIX-02, FIX-03.

    Args  (tất cả giữ nguyên interface bản gốc):
        device       : torch.device
        num_nodes    : 207 (METR-LA) hoặc 325 (PEMS-BAY)
        dropout      : float, default 0.3
        supports     : list of pre-computed adjacency tensors (hoặc None)
        gcn_bool     : dùng GCN hay không
        addaptadj    : dùng adaptive adjacency
        aptinit      : khởi tạo node embeddings (None = random)
        in_dim       : số input features (default 2: speed + time-of-day)
        out_dim      : số bước dự đoán (default 12)
        residual_channels : channels GCN/TCN
        dilation_channels : channels sau dilation
        skip_channels : channels skip connection
        end_channels  : channels cuối
        kernel_size   : kernel/dilation factor (default 2)
        blocks        : số WaveNet blocks
        layers        : số layers mỗi block
    """
    def __init__(
        self,
        device,
        num_nodes: int,
        dropout: float = 0.3,
        supports=None,
        gcn_bool: bool = True,
        addaptadj: bool = True,
        aptinit=None,
        in_dim: int = 2,
        out_dim: int = 12,
        residual_channels: int = 32,
        dilation_channels: int = 32,
        skip_channels: int = 256,
        end_channels: int = 512,
        kernel_size: int = 2,
        blocks: int = 4,
        layers: int = 2,
    ):
        super().__init__()

        self.dropout = dropout
        self.blocks = blocks
        self.layers = layers
        self.gcn_bool = gcn_bool
        self.addaptadj = addaptadj

        # ── Adaptive adjacency (giữ nguyên bản gốc) ──
        if addaptadj:
            if aptinit is None:
                self.nodevec1 = nn.Parameter(
                    torch.randn(num_nodes, 10).to(device), requires_grad=True
                )
                self.nodevec2 = nn.Parameter(
                    torch.randn(10, num_nodes).to(device), requires_grad=True
                )
            else:
                m, p, n = torch.svd(aptinit)
                initemb1 = torch.mm(m[:, :10], torch.diag(p[:10] ** 0.5))
                initemb2 = torch.mm(torch.diag(p[:10] ** 0.5), n[:, :10].t())
                self.nodevec1 = nn.Parameter(initemb1.to(device), requires_grad=True)
                self.nodevec2 = nn.Parameter(initemb2.to(device), requires_grad=True)

        self.supports = supports
        support_len = 0 if supports is None else len(supports)
        if gcn_bool and addaptadj:
            support_len += 1

        # ── Input projection ──
        self.start_conv = nn.Conv2d(
            in_channels=in_dim, out_channels=residual_channels,
            kernel_size=(1, 1)
        )

        # ── FIX-02: CausalWindowAttnTCN thay dilated conv ──
        self.tcn_layers = nn.ModuleList()
        # ── GCN layers ──
        self.gcn_layers = nn.ModuleList()
        # ── Skip projections ──
        self.skip_convs = nn.ModuleList()
        # ── Residual projections ──
        self.residual_convs = nn.ModuleList()
        # ── BN ──
        self.bn = nn.ModuleList()

        receptive_field = 1
        for b in range(blocks):
            new_dilation = 1
            for _ in range(layers):
                # FIX-02: CausalWindowAttnTCN
                self.tcn_layers.append(
                    CausalWindowAttnTCN(
                        in_channels=residual_channels,
                        out_channels=dilation_channels,
                        kernel_size=new_dilation,
                        num_heads=4,
                        dropout=dropout,
                    )
                )
                # Skip projection
                self.skip_convs.append(
                    nn.Conv2d(dilation_channels, skip_channels, kernel_size=(1, 1))
                )
                # GCN
                if gcn_bool:
                    self.gcn_layers.append(
                        gcn(dilation_channels, residual_channels,
                            dropout, support_len=support_len)
                    )
                else:
                    self.residual_convs.append(
                        nn.Conv2d(dilation_channels, residual_channels,
                                  kernel_size=(1, 1))
                    )
                self.bn.append(nn.BatchNorm2d(residual_channels))

                receptive_field += new_dilation
                new_dilation *= 2

        self.receptive_field = receptive_field

        # ── FIX-01: Cross-layer Skip Attention ──
        self.skip_agg_attn = SkipAggregationAttn(
            channels=skip_channels, num_heads=8
        )

        # ── Output layers ──
        self.end_conv_1 = nn.Conv2d(
            skip_channels, end_channels, kernel_size=(1, 1), bias=True
        )
        self.end_conv_2 = nn.Conv2d(
            end_channels, out_dim, kernel_size=(1, 1), bias=True
        )

    def forward(self, input, adj_mx=None):
        """
        input  : (B, in_dim, N, T)   — T thường = 13 (receptive field)
        adj_mx : pre-computed adj (không bắt buộc nếu dùng adaptive)
        returns: (B, out_dim, N, 1)
        """
        # Pad nếu input sequence ngắn hơn receptive field
        in_len = input.size(3)
        if in_len < self.receptive_field:
            pad = self.receptive_field - in_len
            input = F.pad(input, (pad, 0, 0, 0))

        x = self.start_conv(input)

        # Build support list
        new_supports = []
        if self.supports is not None:
            new_supports = list(self.supports)
        if self.gcn_bool and self.addaptadj:
            adp = F.softmax(F.relu(
                torch.mm(self.nodevec1, self.nodevec2)
            ), dim=1)
            new_supports.append(adp)

        skip_list = []
        gcn_idx = 0
        for layer_idx in range(self.blocks * self.layers):
            residual = x

            # FIX-02: Causal Window Attention TCN
            x_tcn = self.tcn_layers[layer_idx](x)

            # Skip connection
            s = self.skip_convs[layer_idx](x_tcn)
            skip_list.append(s)

            # GCN
            if self.gcn_bool:
                x = self.gcn_layers[gcn_idx](x_tcn, new_supports)
                gcn_idx += 1
            else:
                x = self.residual_convs[layer_idx](x_tcn)

            # Residual — align time dimension
            x = x + residual[:, :, :, -x.size(3):]
            x = self.bn[layer_idx](x)

        # FIX-01: Cross-layer Skip Attention thay vì simple sum
        x = self.skip_agg_attn(skip_list)           # (B, skip_channels, N, 1)

        x = F.relu(x)
        x = F.relu(self.end_conv_1(x))
        x = self.end_conv_2(x)                      # (B, out_dim, N, 1)
        return x


# ─────────────────────────────────────────────
#  FIX-03: build_optimizer với per-module LR
# ─────────────────────────────────────────────

def build_optimizer(
    model: gwnet,
    base_lr: float = 0.001,
    attn_lr_multiplier: float = 4.0,
    adj_lr_multiplier: float = 0.5,
    weight_decay: float = 1e-4,
) -> torch.optim.Optimizer:
    """
    FIX-03: Optimizer với learning rate riêng cho từng module.

    - Adaptive adj (nodevec1, nodevec2):  base_lr * adj_lr_multiplier
    - Attention modules (TCN, skip_agg):  base_lr * attn_lr_multiplier
    - Phần còn lại (GCN, linear, BN):     base_lr

    Rationale (từ roadmap FIX-03):
      Nếu gradient bị dominated bởi attention, scale LR attention lên 3-5×.
      Adaptive matrix cần LR nhỏ hơn để ổn định.

    Usage:
        optimizer = build_optimizer(model, base_lr=0.001)
        # hoặc
        optimizer = build_optimizer(model, base_lr=0.001, attn_lr_multiplier=3.0)

    Returns:
        torch.optim.Adam with param_groups
    """
    adj_params, attn_params, base_params = [], [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'nodevec' in name:
            adj_params.append(param)
        elif any(k in name for k in ['tcn_layers', 'skip_agg_attn', 'rel_pe',
                                      'q_proj', 'k_proj', 'v_proj', 'out_proj',
                                      'rel_bias', 'gate']):
            attn_params.append(param)
        else:
            base_params.append(param)

    param_groups = [
        {'params': base_params,  'lr': base_lr,                          'name': 'base'},
        {'params': attn_params,  'lr': base_lr * attn_lr_multiplier,     'name': 'attention'},
        {'params': adj_params,   'lr': base_lr * adj_lr_multiplier,      'name': 'adaptive_adj'},
    ]

    optimizer = torch.optim.Adam(
        param_groups, lr=base_lr, weight_decay=weight_decay
    )
    return optimizer


def print_optimizer_lr(optimizer: torch.optim.Optimizer):
    """Debug helper: in LR của từng group."""
    for g in optimizer.param_groups:
        name = g.get('name', '?')
        n_params = len(g['params'])
        print(f"  [{name:20s}]  lr={g['lr']:.2e}  n_params={n_params}")


# ─────────────────────────────────────────────
#  Gradient monitoring (FIX-03 diagnosis)
# ─────────────────────────────────────────────

def register_grad_monitor(model: gwnet):
    """
    Đăng ký hook để in gradient norm của từng module.
    Gọi 1 lần sau khi tạo model, trước training loop.
    Dùng để phát hiện gradient domination (FIX-03).

    Usage:
        register_grad_monitor(model)
        # train 1 batch rồi xem output
    """
    hooks = []

    def make_hook(name):
        def hook(grad):
            if grad is not None:
                norm = grad.norm().item()
                if norm > 0:
                    print(f"  grad_norm [{name:40s}] = {norm:.4f}")
        return hook

    for name, param in model.named_parameters():
        if param.requires_grad:
            h = param.register_hook(make_hook(name))
            hooks.append(h)

    return hooks   # giữ reference để remove sau: for h in hooks: h.remove()


# ─────────────────────────────────────────────
#  Quick sanity check  (chạy trực tiếp file này)
# ─────────────────────────────────────────────

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    # METR-LA config
    for dataset, num_nodes in [('METR-LA', 207), ('PEMS-BAY', 325)]:
        print(f"\n{'='*50}")
        print(f"Testing {dataset}  (num_nodes={num_nodes})")
        model = gwnet(
            device=device,
            num_nodes=num_nodes,
            dropout=0.3,
            supports=None,
            gcn_bool=True,
            addaptadj=True,
            in_dim=2,
            out_dim=12,
            residual_channels=32,
            dilation_channels=32,
            skip_channels=256,
            end_channels=512,
            kernel_size=2,
            blocks=4,
            layers=2,
        ).to(device)

        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable params: {total_params:,}")

        # Forward pass
        B, T = 4, 13
        x = torch.randn(B, 2, num_nodes, T).to(device)
        with torch.no_grad():
            out = model(x)
        print(f"  Input  shape: {list(x.shape)}")
        print(f"  Output shape: {list(out.shape)}")
        assert out.shape == (B, 12, num_nodes, 1), f"Shape mismatch: {out.shape}"
        print(f"  ✓ Shape OK")

        # Optimizer
        opt = build_optimizer(model, base_lr=1e-3)
        print("  Optimizer param groups:")
        print_optimizer_lr(opt)

    print("\nAll checks passed ✓")

Overwriting /kaggle/working/Graph-WaveNet/model.py


In [6]:
!pip install -r requirements.txt

In [7]:
!rm -rf data/METR-LA data/PEMS-BAY

!python generate_training_data.py \
    --output_dir=data/METR-LA \
    --traffic_df_filename=/kaggle/input/datasets/annnnguyen/metr-la-dataset/METR-LA.h5

!python generate_training_data.py \
    --output_dir=data/PEMS-BAY \
    --traffic_df_filename=/kaggle/input/datasets/scchuy/pemsbay/pems-bay.h5

x shape:  (34249, 12, 207, 2) , y shape:  (34249, 12, 207, 2)
train x:  (23974, 12, 207, 2) y: (23974, 12, 207, 2)
val x:  (3425, 12, 207, 2) y: (3425, 12, 207, 2)
test x:  (6850, 12, 207, 2) y: (6850, 12, 207, 2)
x shape:  (52093, 12, 325, 2) , y shape:  (52093, 12, 325, 2)
train x:  (36465, 12, 325, 2) y: (36465, 12, 325, 2)
val x:  (5209, 12, 325, 2) y: (5209, 12, 325, 2)
test x:  (10419, 12, 325, 2) y: (10419, 12, 325, 2)


In [8]:
# Tạo thư mục lưu checkpoint trước
!mkdir -p garage

In [9]:
!python train.py \
    --device cuda:0 \
    --data data/PEMS-BAY \
    --adjdata data/sensor_graph/adj_mx_bay.pkl \
    --adjtype doubletransition \
    --gcn_bool \
    --addaptadj \
    --num_nodes 325 \
    --save garage/ \
    --expid 2

Traceback (most recent call last):
  File "/kaggle/working/Graph-WaveNet/train.py", line 173, in <module>
    main()
  File "/kaggle/working/Graph-WaveNet/train.py", line 46, in main
    supports = [torch.tensor(i).to(device) for i in adj_mx]
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py", line 417, in _lazy_init
    raise AssertionError("Torch not compiled with CUDA enabled")
AssertionError: Torch not compiled with CUDA enabled


In [10]:
# !python test.py \
#     --device cuda:0 \
#     --data data/METR-LA \
#     --adjdata data/sensor_graph/adj_mx.pkl \
#     --adjtype doubletransition \
#     --gcn_bool \
#     --addaptadj \
#     --num_nodes 207 \
#     --checkpoint /kaggle/input/graohwavenet/_epoch_53_2.79.pth